# Track C — full pipeline (notebook)

Same code as the CLI, driven from here. Every stage reads a YAML config; edit
`configs/**` and `prompts/**`, not this notebook, to change an experiment.

```
download_data -> build_grounding -> build_traces -> train_sft_distill -> train_grpo
                 output/grounding/  output/traces/  output/sft/          output/grpo/
```


## 0. Setup

Run from `bioreason_sft/`. GPU stages need CC >= 7.5 (T4/A100/H100); **P100 will not work**
(bitsandbytes 4-bit needs Turing+, unsloth needs Volta+).

In [ ]:
# !pip install -q "unsloth[kaggle-new]" "trl>=0.9" datasets scikit-learn decoupler requests pyyaml kaggle

import os, sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
os.environ.setdefault("MLGENX_ROOT", str(Path.cwd().parent))

import paths, config as cfgmod
paths.describe()

## 1. Data
Needs `~/.kaggle/kaggle.json` (or `KAGGLE_USERNAME`/`KAGGLE_KEY`) **and** you must have
accepted the competition rules on the website.

In [ ]:
!python download_data.py

## 2. Grounding

mygene.info (gene function) + CollecTRI (**signed** TF->target edges). ~5 min, no API key.

CollecTRI is the highest-value source because the sign gives direction: knock down an
activator -> target **down**; knock down a repressor -> target **up**. That is DIR-AUROC,
the metric half even SOTA only reaches ~0.65-0.73 on.

Grounding is **train-time only** — the student never sees it (Track C forbids tools at inference).

In [ ]:
GROUNDING_RUN = "default"
!python build_grounding.py --out {GROUNDING_RUN}

In [ ]:
# Check coverage BEFORE spending on traces.
g = json.loads(paths.grounding_json(GROUNDING_RUN).read_text())
print(g["stats"])
# Low annotation coverage -> the 'lantern effect' bites: the teacher has little to
# reason from on poorly-characterised mouse genes and may confabulate.
print("\n--- example context block ---")
print(next(iter(g["rows"].values()))["context"][:800])

## 3. Reasoning traces

SynthPert **Approach 2**: the teacher is *given* the label and asked to *rationalize* it,
so the teacher's own accuracy stops mattering (o4-mini scored ~52% on this task, yet its
traces trained an 8B student to ~89%).

**Always smoke-test first** — 30 rows costs cents and tells you what 2000 would waste.

In [ ]:
os.environ["TEACHER_API_KEY"] = ""   # <-- set your key
assert os.environ["TEACHER_API_KEY"], "set TEACHER_API_KEY"

!python build_traces.py --out smoke --config smoke

In [ ]:
# READ THE TRACES. Highest-leverage QA in the pipeline.
# Looking for SPECIFIC gene-level function ('Rpf2 is a 60S maturation factor...')
# vs vague pathway hand-waving ('both involved in immune response').
# The former generalises; the latter correlates with WRONG answers.
for line in paths.traces_jsonl("smoke").read_text().splitlines()[:5]:
    r = json.loads(line)
    print(f"=== {r['id']}  label={r['label']}  critic={r['critic_score']}")
    print(r["reasoning"][:700], "\n")

In [ ]:
# Full run: ~1-3h at 8 workers. Resumable — re-running skips ids already written.
TRACES_RUN = "default"
!python build_traces.py --out {TRACES_RUN} --config default

# variants without touching code:
# !python build_traces.py --out terse --set prompts=teacher/terse
# !python build_traces.py --out gpt   --set teacher.model=gpt-5 --set teacher.base_url=https://api.openai.com/v1
# !python build_traces.py --out loose --set critic.min_score=4

In [ ]:
import pandas as pd
tdf = pd.read_json(paths.traces_jsonl(TRACES_RUN), lines=True)
print(len(tdf), "traces"); print(tdf["label"].value_counts().to_dict())
print(json.loads((paths.run_dir("traces", TRACES_RUN)/"stats.json").read_text()))

## 4. SFT (QLoRA + reasoning distillation)

The **treatment**. Label-only SFT is the control and barely moves (~0.59 DE-AUROC vs 0.58
baseline; *below* chance on direction). The delta is the data, not the method.

Validation is a **two-axis blocked split** (perturbations *and* genes held out), matching the
real test design. ~30% of rows get discarded — that is the correct price.

In [ ]:
SFT_RUN = "qwen4b"
SFT_CONFIG = "default"      # or "h100_8b" on the cluster

!python train_sft_distill.py --config {SFT_CONFIG} --traces {TRACES_RUN} --out {SFT_RUN}

# resume after an interruption (safe on a fresh run too):
# !python train_sft_distill.py --config {SFT_CONFIG} --traces {TRACES_RUN} --out {SFT_RUN} --resume
# one-off override without editing yaml:
# !python train_sft_distill.py --out r64 --traces {TRACES_RUN} --set lora.r=64 --set train.epochs=2

In [ ]:
m = json.loads((paths.sft_dir(SFT_RUN)/"metrics.json").read_text())
print(json.dumps(m, indent=2))
print("\n--- reasoning generated on held-out perturbations ---")
print((paths.sft_dir(SFT_RUN)/"val_reasoning_samples.txt").read_text()[:1200])

## 5. GRPO

Warm-started from the SFT adapter — **required**: RL alone does not add new reasoning priors;
SFT on traces adds the primitives RL then explores.

**Watch `val/score`, not the reward.** Correctness-only RL sharpens toward argmax, scores tie,
and a rank metric craters *while reward climbs*. `grpo.beta` (KL) is the guard.

In [ ]:
GRPO_RUN = f"{SFT_RUN}-grpo"
GRPO_CONFIG = "default"     # or "h100"

!python train_grpo.py --config {GRPO_CONFIG} --sft {SFT_RUN} --out {GRPO_RUN}

# ablation: isolate the CollecTRI verifier
# !python train_grpo.py --config no_verifier --sft {SFT_RUN} --out {SFT_RUN}-noverif
# stronger KL if the val score collapses:
# !python train_grpo.py --sft {SFT_RUN} --out {SFT_RUN}-beta10 --set grpo.beta=0.1

In [ ]:
gm = json.loads((paths.grpo_dir(GRPO_RUN)/"metrics.json").read_text())
print("best blocked-val:", gm["best_blocked_val"])
print("verifier direction acc:", gm["verifier_direction_acc"])

import matplotlib.pyplot as plt
h = pd.DataFrame(gm["history"])
if len(h):
    h.plot(x="step", y=["de","dir","score"], marker="o", figsize=(7,4),
           title="blocked-val during GRPO (falling score + rising reward = collapse)")
    plt.axhline(0.5, ls="--", c="grey"); plt.show()

## 6. Compare runs
Every run snapshots its fully-resolved config, so the table is self-documenting.

In [ ]:
rows = []
for stage in ("sft","grpo"):
    d = paths.OUTPUT/stage
    for run in sorted([x for x in d.iterdir() if x.is_dir()] if d.exists() else []):
        mp = run/"metrics.json"
        if not mp.exists(): continue
        m = json.loads(mp.read_text())
        rows.append({"stage": stage, "run": run.name,
                     "score": m.get("blocked_val_score") or m.get("best_blocked_val"),
                     "traces": m.get("traces_run"), "from": m.get("sft_run"),
                     "n_traces": m.get("n_traces")})
pd.DataFrame(rows).sort_values("score", ascending=False)